In [ ]:
import os
import sys
import torch

# Lấy đường dẫn thư mục gốc của dự án (thư mục cha của notebooks)
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))

# Thêm đường dẫn này vào hệ thống để Python tìm thấy các module
if PROJECT_ROOT not in sys.path:
    sys.path.append(PROJECT_ROOT)

# Bây giờ mới gọi các module của bạn
from models.models import BaseCMNModel
print("Import thành công!")


In [4]:
import torch
from models.models import BaseCMNModel

class Args:
    dataset_name = 'iu_xray'
    visual_extractor = 'resnet101'
    visual_extractor_pretrained = False
    d_model = 512
    d_ff = 512
    d_vf = 2048
    num_heads = 8
    num_layers = 3
    dropout = 0.1
    drop_prob_lm = 0.5
    max_seq_length = 60
    use_bn = 0
    pad_idx = 0
    bos_idx = 1
    eos_idx = 2
    topk = 32
    cmm_size = 2048
    cmm_dim = 512
    # Thêm các tham số khác nếu cần...

def test():
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    # Giả lập tokenizer
    class Tokenizer:
        def __init__(self):
            self.vocab_size = 760
            self.idx2token = {0: '<pad>', 1: '<bos>', 2: '<eos>', 3: '<unk>'}
            self.idx2token.update({i: f'token_{i}' for i in range(4, self.vocab_size)})
            self.token2idx = {token: idx for idx, token in self.idx2token.items()}
            self.pad_idx = 0
            self.bos_idx = 1
            self.eos_idx = 2
            self.unk_idx = 3
    
    tokenizer = Tokenizer()
    args = Args()
    
    print("--- Đang khởi tạo mô hình với Swin Encoder... ---")
    model = BaseCMNModel(args, tokenizer).to(device)
    
    # Tạo dữ liệu giả (Batch size 2, 2 ảnh, 3 kênh, 224x224)
    dummy_images = torch.randn(2, 2, 3, 224, 224).to(device)
    dummy_targets = torch.randint(4, 760, (2, args.max_seq_length)).to(device)
    dummy_targets[:, 0] = args.bos_idx
    
    print("--- Đang chạy thử Forward pass (Training mode)... ---")
    try:
        with torch.no_grad():
            output = model(dummy_images, dummy_targets, mode='train')
        print(f"Thành công! Output shape: {output.shape}")
        print("Kiến trúc Encoder hiện tại:")
        print(model.encoder_decoder.model.encoder)
    except Exception as e:
        print(f"Lỗi rồi: {e}")

if __name__ == "__main__":
    test()


--- Đang khởi tạo mô hình với Swin Encoder... ---
--- Đang chạy thử Forward pass (Training mode)... ---
Thành công! Output shape: torch.Size([2, 59, 761])
Kiến trúc Encoder hiện tại:
SwinEncoder(
  (layers): ModuleList(
    (0-2): 3 x SwinBlock(
      (norm1): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
      (attn): MultiheadAttention(
        (out_proj): NonDynamicallyQuantizableLinear(in_features=512, out_features=512, bias=True)
      )
      (norm2): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
      (ffn): Sequential(
        (0): Linear(in_features=512, out_features=2048, bias=True)
        (1): GELU(approximate='none')
        (2): Linear(in_features=2048, out_features=512, bias=True)
      )
    )
  )
  (norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
)
